# 날짜 귀속 정량 비교 — baseline(모델이 날짜계산) vs 뉴로-심볼릭(코드가 계산)

같은 RunPod **base 모델**(Qwen2.5-7B, LoRA 없음)에 두 경로를 같은 프롬프트셋으로 돌려 **task별 날짜 정확도**를 잰다.

## 문제 정의

"장보고 운동해야지 **내일은** 친구들 만나기로 했어" 같은 입력을 (1) task로 쪼개고, (2) 각 task에 **정확한 절대날짜**를 붙여야 한다 → 장보기·운동=오늘, 친구 약속=내일.

base 모델에게 `due_date`(절대날짜)를 직접 시켰더니 두 부류의 오류가 났다:

- **정규화 오류** — "내일"을 +3일(엉뚱한 날짜)로 _계산_ 실수
- **귀속 오류** — "내일"을 친구 약속이 아닌 장보기/운동에 잘못 _연결_

이건 NLP에서 오래된 두 표준 하위문제다: **TIMEX 정규화**(상대표현→절대날짜)와 **event–time linking**(이벤트↔시간표현 연결, TempEval의 TLINK).

## 두 방법론

### baseline (현재 프로덕션) — `pipeline.run`

모델이 한 번에 `{title, due_date(YYYY-MM-DD), tags}`를 출력. **분리 + 날짜 계산 + 귀속을 전부 모델이** 한다. force-today 가드(시간표현이 전혀 없으면 today로 강제)만 코드가 보정.

### neuro-symbolic (신규) — `llm.split_tasks` + `when_resolver`

**"뉴로"(신경망=LLM)와 "심볼릭"(규칙 코드)으로 역할을 쪼갠 것**이 핵심이다.

```
[뉴로 / LLM]   split_tasks:
   입력 → task별 {title, when, tags}
   when = 입력에 등장한 시간표현 '구문' 그대로 ("내일","이번주","사흘 뒤") 또는 null
   ※ 모델은 절대날짜를 계산하지 않는다. "어느 task에 어느 시간표현이 붙나"(귀속)만 판단.
        ↓
[심볼릭 / 코드]  when_resolver.resolve_when(when, today):
   "내일"→today+1, "이번주"→이번주 일요일, "사흘 뒤"→today+3,
   "다음주"→today+7, "6월21일"/ISO→그 날짜, null→today, 과거→today 클램프
```

**왜 이렇게?** 날짜 *계산*은 결정적이라 코드가 100% 맞춘다 → 정규화 오류 부류가 **구조적으로 0**이 된다. 모델에는 더 쉬운 일(분리·귀속)만 남긴다. 프로덕션 비용은 동일(LLM 1회 호출).

## 측정 지표

각 case의 gold task(제목 부분일치 key)별로 날짜가 정답과 같은지 센다.

- **per-task 날짜 정확도** = 맞은 task 수 / 전체 gold task 수
- **완전정답 case** = 모든 task의 날짜가 맞은 프롬프트 수

정답(gold)은 `resolver`를 쓰지 않고 plain `datetime`으로 **독립 계산**한 오라클이다(cell-3). neuro에서 남는 오류는 대부분 **귀속**(모델이 시간표현을 엉뚱한 task에 붙임) 또는 **task 환각**(없는 일을 지어냄)이며, 그게 이 비교가 드러내려는 지점이다.

> **전환 완료(2026-06-18):** 이 비교로 neuro 우위(62.5%→93.8%)를 확인한 뒤 `split_tasks` 기본 경로를 뉴로-심볼릭으로 전환했다. 따라서 지금은 baseline(`run`)도 neuro 경로와 동일하다. 아래 저장된 출력이 전환 전 비교 기록이다. (ADR-0006)


In [18]:
import sys
import pathlib
import os

_root = pathlib.Path.cwd()
while not (_root / "agents").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_env = _root / ".env"
if _env.is_file():
    for line in _env.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

print("repo root =", _root, "| LLM_PROVIDER =", os.environ.get("LLM_PROVIDER"))

repo root = /Users/jpaper/Documents/projects/mong-studio/mongle-ai | LLM_PROVIDER = runpod


In [19]:
from datetime import date, datetime, timedelta

from api.config import AppConfig
from api.deps import build_todo_generate_ports
from agents.todo_creation.schemas import TodoInput, CandidatesResult
from agents.todo_creation.todo.pipeline import run

cfg = AppConfig.from_env()
ports = build_todo_generate_ports(cfg)
TODAY = date.today()
print(
    "ports:",
    type(ports.llm).__name__,
    "| TODAY:",
    TODAY,
    "(" + "월화수목금토일"[TODAY.weekday()] + ")",
)

ports: RunPodQwenLLM | TODAY: 2026-06-18 (목)


In [20]:
# ── gold 데이터셋 (파일 로드 없이 이 노트북에 직접 명시) ──
# 정답은 resolver 를 import 하지 않고 plain datetime 으로 독립 계산한다(오라클).
# spec 의미: today=오늘, d1~d3=오늘+N, sunday=이번주 일요일, next_week=오늘+7일.
def gold_date(spec, today=None):
    today = today or TODAY
    wd = today.weekday()
    if spec == "fri_thisweek":
        d = today + timedelta(days=4 - wd)
        return d if d >= today else today
    return {
        "today": today,
        "d1": today + timedelta(days=1),
        "d2": today + timedelta(days=2),
        "d3": today + timedelta(days=3),
        "sunday": today + timedelta(days=6 - wd),
        "next_week": today + timedelta(days=7),  # "다음주" = 오늘 기준 일주일 뒤
    }[spec]


# (제목 부분일치 key, 기대 날짜 spec)
CASES = [
    {
        "prompt": "장보고 운동해야지 내일은 친구들 만나기로 했어",
        "gold": [("장보", "today"), ("운동", "today"), ("친구", "d1")],
    },
    {
        "prompt": "과제 제출하고 장보러 가기",
        "gold": [("과제", "today"), ("장보", "today")],
    },
    {"prompt": "내일 토익 시험", "gold": [("토익", "d1")]},
    {"prompt": "이번주 안으로 과제 제출하기", "gold": [("과제", "sunday")]},
    {"prompt": "모레 치과 예약하기", "gold": [("치과", "d2")]},
    {"prompt": "3일 뒤 발표", "gold": [("발표", "d3")]},
    {
        "prompt": "오늘 빨래하고 내일 청소하기",
        "gold": [("빨래", "today"), ("청소", "d1")],
    },
    {
        "prompt": "장보고 내일 친구 만나고 모레 병원 가기",
        "gold": [("장보", "today"), ("친구", "d1"), ("병원", "d2")],
    },
    {"prompt": "이번주 금요일까지 보고서 제출", "gold": [("보고서", "fri_thisweek")]},
    {"prompt": "다음주 워크숍 참석", "gold": [("워크숍", "next_week")]},
    # 동의어/문어체/순우리말 커버리지
    {"prompt": "금일 은행 업무 보기", "gold": [("은행", "today")]},
    {"prompt": "명일 회의 자료 정리", "gold": [("회의", "d1")]},
    {"prompt": "사흘 뒤 출장", "gold": [("출장", "d3")]},
]
print(len(CASES), "cases |", sum(len(c["gold"]) for c in CASES), "gold tasks")

13 cases | 19 gold tasks


In [21]:
def _tasks_from_run(res):
    if not isinstance(res, CandidatesResult):
        return []
    return [(t.title, t.due_date) for t in (res.todos + res.calendar_events)]


async def baseline(prompt):
    try:
        res = await run(
            TodoInput(user_id="eval", prompt=prompt, today=TODAY),
            ports=ports,
            now=datetime.now(),
        )
        return _tasks_from_run(res)
    except Exception as e:  # 한 케이스 실패가 전체 평가를 죽이지 않게
        print(f"  ! baseline 실패: {prompt[:20]}… {type(e).__name__}")
        return []


async def neuro(prompt):
    try:
        sr = await ports.llm.split_tasks(prompt=prompt, today=TODAY)
        return [(t.title, t.due_date) for t in sr.tasks]
    except Exception as e:
        print(f"  ! neuro 실패: {prompt[:20]}… {type(e).__name__}")
        return []


def score(pred, gold):
    correct = 0
    for key, spec in gold:
        exp = gold_date(spec)
        hit = next((d for (title, d) in pred if key in title), None)
        correct += int(hit == exp)
    return correct

In [22]:
# 두 경로를 병렬 실행 — 케이스 10개 × (baseline+neuro) = 20호출을 동시에
# (순차로 돌리면 RunPod 폴링/콜드스타트가 호출마다 직렬로 쌓여 매우 느리다)
import asyncio

# 동시 호출 상한 — RunPod 동시 워커/레이트리밋에 걸리면 숫자를 낮춰라
_SEM = asyncio.Semaphore(8)


async def _guarded(coro_fn, prompt):
    async with _SEM:
        return await coro_fn(prompt)


async def run_case(c):
    b, n = await asyncio.gather(
        _guarded(baseline, c["prompt"]), _guarded(neuro, c["prompt"])
    )
    g = len(c["gold"])
    bc, nc = score(b, c["gold"]), score(n, c["gold"])
    print(f"[neuro {nc}/{g} | base {bc}/{g}]  {c['prompt']}")
    return {"prompt": c["prompt"], "g": g, "bc": bc, "nc": nc, "b": b, "n": n}


rows = await asyncio.gather(*(run_case(c) for c in CASES))
print("\ndone")


[todo_creation] start  kind=generate  user=eval
  input         : 장보고 운동해야지 내일은 친구들 만나기로 했어

[todo_creation] start  kind=generate  user=eval
  input         : 과제 제출하고 장보러 가기

[todo_creation] start  kind=generate  user=eval
  input         : 내일 토익 시험

[todo_creation] start  kind=generate  user=eval
  input         : 이번주 안으로 과제 제출하기
[STEP 1] validate
[STEP 1] validate
[STEP 1] validate
[STEP 1] validate

[todo_creation] start  kind=generate  user=eval
  input         : 모레 치과 예약하기
[STEP 1] validate
[STEP 2] task_splitter
  split_tasks       : 1 items
    [1] 토익 시험 | 2026-06-19
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[] calendar_events=[TaskCandidate(title='토익 시험', due_date=datetime.date(2026, 6, 19), tags=['학습'])] summary_text=None profile_memory_patch=None
[todo_creation] done


[todo_creation] start  kind=generate  user=eval
  input         : 3일 뒤 발표
[STEP 1] validate


[neuro 1/1 | base 1/1]  내일 토익 시험


[STEP 2] task_splitter
  split_tasks       : 3 items
    [1] 장보기 | 2026-06-19
    [2] 운동가기 | 2026-06-19
    [3] 친구들 만나기 | 2026-06-21
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[] calendar_events=[TaskCandidate(title='장보기', due_date=datetime.date(2026, 6, 19), tags=['일상']), TaskCandidate(title='운동가기', due_date=datetime.date(2026, 6, 19), tags=['건강']), TaskCandidate(title='친구들 만나기', due_date=datetime.date(2026, 6, 21), tags=['일상'])] summary_text=None profile_memory_patch=None
[todo_creation] done

[STEP 2] task_splitter
  split_tasks       : 1 items
    [1] 과제 제출 | 2026-06-21
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[] calendar_events=[TaskCandidate(title='과제 제출', due_date=datetime.date(2026, 6, 21), tags=['학습'])] summary_text=None profile_memory_patch=None
[todo_creation] done


[todo_creation] start  kind=generate  user=eval
  input         : 오늘 빨래하고 내일 청소하기
[STEP 1] validate


[neuro 1/1 | base 1/1]  이번주 안으로 과제 제출하기


[STEP 2] task_splitter
  split_tasks       : 2 items
    [1] 과제 제출 | 2026-06-18
    [2] 장보기 | 2026-06-18
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[TaskCandidate(title='과제 제출', due_date=datetime.date(2026, 6, 18), tags=['학습']), TaskCandidate(title='장보기', due_date=datetime.date(2026, 6, 18), tags=['일상'])] calendar_events=[] summary_text=None profile_memory_patch=None
[todo_creation] done


[todo_creation] start  kind=generate  user=eval
  input         : 장보고 내일 친구 만나고 모레 병원 가기
[STEP 1] validate


...

[neuro 1/1 | base 1/1]  금일 은행 업무 보기


[STEP 2] task_splitter
  split_tasks       : 1 items
    [1] 회의 자료 정리 | 2026-06-18
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[TaskCandidate(title='회의 자료 정리', due_date=datetime.date(2026, 6, 18), tags=['업무'])] calendar_events=[] summary_text=None profile_memory_patch=None
[todo_creation] done



[neuro 1/1 | base 0/1]  명일 회의 자료 정리


[STEP 2] task_splitter
  split_tasks       : 1 items
    [1] 출장 | 2026-06-18
[STEP 3] date_router
  result            : kind='candidates' thread_id='' todos=[TaskCandidate(title='출장', due_date=datetime.date(2026, 6, 18), tags=['업무'])] calendar_events=[] summary_text=None profile_memory_patch=None
[todo_creation] done



[neuro 0/1 | base 0/1]  사흘 뒤 출장

done


In [23]:
# ── 정량 결과 ──
total = sum(r["g"] for r in rows)
base = sum(r["bc"] for r in rows)
neu = sum(r["nc"] for r in rows)
base_full = sum(r["bc"] == r["g"] for r in rows)
neu_full = sum(r["nc"] == r["g"] for r in rows)

print("=" * 60)
print("per-task 날짜 정확도")
print(f"  baseline (모델이 날짜계산) : {base}/{total} = {base / total:.1%}")
print(f"  neuro    (코드가 날짜계산) : {neu}/{total} = {neu / total:.1%}")
print(f"  delta                      : {(neu - base) / total:+.1%}")
print("-" * 60)
print("프롬프트 완전정답(모든 task 정확)")
print(f"  baseline : {base_full}/{len(rows)}")
print(f"  neuro    : {neu_full}/{len(rows)}")
print("=" * 60)


def fmt(pred):
    return ", ".join(f"{t}:{d.isoformat()[5:]}" for t, d in pred) or "-"


for r in rows:
    flag = "✓" if r["nc"] == r["g"] else "✗"
    print(f"\n{flag} {r['prompt']}  (gold {r['g']})")
    print(f"   base [{r['bc']}/{r['g']}] {fmt(r['b'])}")
    print(f"   neuro[{r['nc']}/{r['g']}] {fmt(r['n'])}")

per-task 날짜 정확도
  baseline (모델이 날짜계산) : 12/19 = 63.2%
  neuro    (코드가 날짜계산) : 14/19 = 73.7%
  delta                      : +10.5%
------------------------------------------------------------
프롬프트 완전정답(모든 task 정확)
  baseline : 8/13
  neuro    : 8/13

✗ 장보고 운동해야지 내일은 친구들 만나기로 했어  (gold 3)
   base [0/3] 장보기:06-19, 운동가기:06-19, 친구들 만나기:06-21
   neuro[2/3] 장보기:06-18, 운동가기:06-18, 친구들 만나기:06-21

✗ 과제 제출하고 장보러 가기  (gold 2)
   base [2/2] 과제 제출:06-18, 장보기:06-18
   neuro[1/2] 과제 제출:06-21, 장보기:06-18

✓ 내일 토익 시험  (gold 1)
   base [1/1] 토익 시험:06-19
   neuro[1/1] 토익 시험:06-19

✓ 이번주 안으로 과제 제출하기  (gold 1)
   base [1/1] 과제 제출:06-21
   neuro[1/1] 과제 제출:06-21

✓ 모레 치과 예약하기  (gold 1)
   base [1/1] 치과 예약하기:06-20
   neuro[1/1] 치과 예약하기:06-20

✓ 3일 뒤 발표  (gold 1)
   base [1/1] 발표:06-21
   neuro[1/1] 발표:06-21

✓ 오늘 빨래하고 내일 청소하기  (gold 2)
   base [2/2] 빨래:06-18, 청소:06-19
   neuro[2/2] 빨래:06-18, 청소:06-19

✗ 장보고 내일 친구 만나고 모레 병원 가기  (gold 3)
   base [2/3] 장보기:06-19, 친구 만나기:06-19, 병원 가기:06-20
   neuro[2/3] 장보기:06-19,

In [24]:
# ── out_of_scope 분류 확인 ──
# 날씨·잡담·감정 표현은 plan 으로 쪼개면 안 된다.
# neuro(split_tasks) 는 intent="out_of_scope", 엔드투엔드(run) 는 kind="out_of_scope" 여야 한다.
OOS_CASES = ["배고프다", "오늘 날씨 어때?", "졸려 죽겠다", "안녕 반가워", "기분이 좋아"]


async def _oos_neuro(prompt):
    try:
        sr = await ports.llm.split_tasks(prompt=prompt, today=TODAY)
        return sr.intent == "out_of_scope"
    except Exception as e:
        print(f"  ! neuro 실패: {prompt} {type(e).__name__}")
        return False


async def _oos_run(prompt):
    try:
        res = await run(
            TodoInput(user_id="eval", prompt=prompt, today=TODAY),
            ports=ports,
            now=datetime.now(),
        )
        return res.kind == "out_of_scope"
    except Exception as e:
        print(f"  ! run 실패: {prompt} {type(e).__name__}")
        return False


oos = await asyncio.gather(
    *[asyncio.gather(_oos_neuro(p), _oos_run(p)) for p in OOS_CASES]
)
ok = sum(n and r for n, r in oos)
print(f"out_of_scope 정확 분류: {ok}/{len(OOS_CASES)}")
for (n, r), p in zip(oos, OOS_CASES):
    flag = "✓" if (n and r) else "✗"
    print(f"  {flag} {p}  (neuro intent={n}, run kind={r})")


[todo_creation] start  kind=generate  user=eval
  input         : 배고프다

[todo_creation] start  kind=generate  user=eval
  input         : 오늘 날씨 어때?

[todo_creation] start  kind=generate  user=eval
  input         : 졸려 죽겠다

[todo_creation] start  kind=generate  user=eval
  input         : 안녕 반가워

[todo_creation] start  kind=generate  user=eval
  input         : 기분이 좋아
[STEP 1] validate
[STEP 1] validate
[STEP 1] validate
[STEP 1] validate
[STEP 1] validate
[STEP 2] task_splitter
[STEP 3] out_of_scope
  result            : kind='out_of_scope' thread_id='' message='나는 목표를 TODO랑 일정으로 차근차근 나눠주는 이장님이야. 준비할 일이나 이루고 싶은 목표를 말해주면 같이 계획을 짜볼게.'
[todo_creation] done

[STEP 2] task_splitter
[STEP 3] out_of_scope
  result            : kind='out_of_scope' thread_id='' message='나는 목표를 TODO랑 일정으로 차근차근 나눠주는 이장님이야. 준비할 일이나 이루고 싶은 목표를 말해주면 같이 계획을 짜볼게.'
[todo_creation] done

[STEP 2] task_splitter
[STEP 3] out_of_scope
  result            : kind='out_of_scope' thread_id='' message='나는 목표를 TODO랑 일정으로 차근차근 나

out_of_scope 정확 분류: 5/5
  ✓ 배고프다  (neuro intent=True, run kind=True)
  ✓ 오늘 날씨 어때?  (neuro intent=True, run kind=True)
  ✓ 졸려 죽겠다  (neuro intent=True, run kind=True)
  ✓ 안녕 반가워  (neuro intent=True, run kind=True)
  ✓ 기분이 좋아  (neuro intent=True, run kind=True)
